# 🧹 Systematic Data Cleaning & Quality Audit Pipeline
**Oasis Infobyte Internship — Data Analytics Track (Level 1 - Task 3)**  
**Author:** Oasis Intern  
**Repository:** `OIBSIP`

---

## 📌 1. Project Overview & Objectives
Real-world data is inherently noisy, containing missing entries, duplicate observations, structural format discrepancies, sensor anomalies, and data type mismatches. This project builds an automated, robust data cleaning pipeline that transforms an uncurated raw dataset into an analytics-ready asset while documenting every analytical decision.

### 📋 Feature Checklist:
- [x] Ingest messy raw dataset and generate an **Initial Data Quality Audit Report**
- [x] Identify and remove exact duplicate rows with documentation
- [x] Standardize string whitespace, character casings, and categorical labels (`Gender`, `Department`)
- [x] Parse mixed-format currency strings into numerical floating-point values
- [x] Parse multiple datetime formats into ISO standard `YYYY-MM-DD`
- [x] Detect and treat numerical outliers and anomalies using domain boundaries and the **Interquartile Range (IQR) Method**
- [x] Strategically handle missing data using median imputation for continuous features and mode for categorical features
- [x] Enforce strict data types and schema integrity
- [x] Produce a comprehensive **"Before vs. After" Data Quality Summary Table**
- [x] Export the cleansed dataset to `data/cleaned_dataset.csv`


In [ ]:
# Environment Setup
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print("Libraries loaded.")


## 🔍 2. Raw Ingestion & Initial Data Quality Report
We load the raw data file and evaluate its state of hygiene before applying any transformations.


In [ ]:
# Load raw dataset
df_raw = pd.read_csv('data/raw_dirty_dataset.csv')
print(f"Raw Dataset Shape: {df_raw.shape[0]} Rows, {df_raw.shape[1]} Columns")
print(f"Total Exact Duplicate Rows: {df_raw.duplicated().sum()}")
df_raw.head(10)


In [ ]:
# Initial Null Values & Data Types Audit
initial_audit = pd.DataFrame({
    'Data_Type': df_raw.dtypes,
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'Unique_Values': df_raw.nunique()
})
initial_audit


## 🧼 3. Step-by-Step Data Cleaning Pipeline
We clone the raw dataframe and execute targeted sanitization stages.


In [ ]:
df = df_raw.copy()

# Step 1: Duplicate Elimination
initial_count = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Step 1 Complete: Dropped {initial_count - len(df)} duplicate records. Current rows: {len(df)}")


### 🔤 Step 2: Whitespace & String Normalization
Trimming whitespace and enforcing proper casing across identifier and textual fields.


In [ ]:
for col in ['Customer_ID', 'Full_Name', 'Gender', 'Department']:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

df['Customer_ID'] = df['Customer_ID'].str.upper()
df['Full_Name'] = df['Full_Name'].str.title()
df[['Customer_ID', 'Full_Name']].head()


### 🏷 Step 3: Categorical Standardization
Unifying inconsistent representations for `Gender` and `Department`.


In [ ]:
gender_map = {
    'MALE': 'Male', 'Male': 'Male', 'male': 'Male', 'M': 'Male',
    'FEMALE': 'Female', 'Female': 'Female', 'female': 'Female', 'F': 'Female',
    'OTHER': 'Other', 'Other': 'Other', 'Unknown': np.nan
}
df['Gender'] = df['Gender'].map(gender_map)

dept_map = {
    'Sales': 'Sales', 'sales': 'Sales',
    'Engineering': 'Engineering', 'ENGINEERING': 'Engineering',
    'Marketing': 'Marketing', 'mktg': 'Marketing',
    'Human Resources': 'Human Resources', 'HR': 'Human Resources',
    'Operations': 'Operations', 'Ops': 'Operations'
}
df['Department'] = df['Department'].map(dept_map)

print("Unique Genders:", df['Gender'].unique())
print("Unique Departments:", df['Department'].unique())


### 💵 Step 4: Currency String Parsing
Extracting pure numerical values from financial strings containing symbols and commas.


In [ ]:
def parse_currency(val):
    if pd.isna(val):
        return np.nan
    s = str(val).replace('$', '').replace(',', '').replace('USD', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

df['Annual_Income'] = df['Annual_Income'].apply(parse_currency)
print("Parsed Annual Income Sample:")
df['Annual_Income'].dropna().head()


### 📅 Step 5: Datetime Normalization
Converting heterogeneous date formats into standard ISO `YYYY-MM-DD`.


In [ ]:
df['Join_Date'] = pd.to_datetime(df['Join_Date'], format='mixed', errors='coerce')
print("Standardized Join Dates:")
df['Join_Date'].dropna().head()


### 📊 Step 6: Outlier Detection & IQR Treatment
Detecting biologically impossible age records (< 18 or > 80), negative incomes, and extreme upper-tail salary spikes using the Interquartile Range ($IQR = Q_3 - Q_1$).


In [ ]:
# Age boundary check
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df.loc[(df['Age'] < 18) | (df['Age'] > 80), 'Age'] = np.nan

# Income IQR outlier capping
df.loc[df['Annual_Income'] <= 0, 'Annual_Income'] = np.nan
income_valid = df['Annual_Income'].dropna()

q1 = income_valid.quantile(0.25)
q3 = income_valid.quantile(0.75)
iqr = q3 - q1
lower_bound = max(0, q1 - 1.5 * iqr)
upper_bound = q3 + 1.5 * iqr

print(f"Income IQR Range: [${lower_bound:,.2f}, ${upper_bound:,.2f}]")
df['Annual_Income'] = df['Annual_Income'].clip(lower=lower_bound, upper=upper_bound)

# Performance score bounds
df['Performance_Score'] = pd.to_numeric(df['Performance_Score'], errors='coerce')
df.loc[(df['Performance_Score'] < 1.0) | (df['Performance_Score'] > 5.0), 'Performance_Score'] = np.nan


### 🧩 Step 7: Missing Value Imputation
Applying statistically sound imputation policies:
- **Numerical** (`Age`, `Annual_Income`, `Performance_Score`): Median (robust to distribution skew).
- **Categorical** (`Gender`, `Department`): Mode (most frequent category).
- **Identifier** (`Customer_ID`): Deterministic synthetic keys.
- **Dates** (`Join_Date`): Median datetime.


In [ ]:
# Reconstruct missing IDs
for idx in df[df['Customer_ID'].isna()].index:
    df.loc[idx, 'Customer_ID'] = f"CUST_{2000 + idx}"

df['Full_Name'] = df['Full_Name'].fillna('Unknown Client')
df['Age'] = df['Age'].fillna(df['Age'].median()).round().astype(int)
df['Annual_Income'] = df['Annual_Income'].fillna(df['Annual_Income'].median()).round(2)
df['Performance_Score'] = df['Performance_Score'].fillna(df['Performance_Score'].median()).round(1)

df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Department'] = df['Department'].fillna(df['Department'].mode()[0])
df['Join_Date'] = df['Join_Date'].fillna(df['Join_Date'].median()).dt.strftime('%Y-%m-%d')

print("Missing values after imputation:")
print(df.isnull().sum())


## 📁 4. Export Cleansed Dataset & Final Quality Report
Saving cleaned asset to CSV and displaying the Before vs. After comparison summary.


In [ ]:
# Save clean CSV
df.to_csv('data/cleaned_dataset.csv', index=False)
print("Saved clean file to data/cleaned_dataset.csv")

# Before vs After Summary Matrix
comparison_records = []
for col in df_raw.columns:
    comparison_records.append({
        'Feature': col,
        'Raw Missing': df_raw[col].isnull().sum(),
        'Clean Missing': df[col].isnull().sum(),
        'Raw Data Type': str(df_raw[col].dtype),
        'Clean Data Type': str(df[col].dtype),
        'Audit Status': '✅ Cleansed & Validated'
    })

audit_df = pd.DataFrame(comparison_records)
audit_df
